# 04 — CNN-1D sobre ULB
Entrena sobre ULB con GPU, 5 corridas, media ± desviacion.
Requiere el 01. Corre todas las celdas.

In [ ]:
import os, datetime as _dt

def iniciar_ejecucion(nombre, config=None, base="resultados"):
    os.makedirs(base, exist_ok=True)
    nums = []
    for d in os.listdir(base):
        if d.startswith("ejecucion_") and os.path.isdir(os.path.join(base, d)):
            try: nums.append(int(d.split("_")[1]))
            except: pass
    n = max(nums)+1 if nums else 1
    carpeta = os.path.join(base, f"ejecucion_{n}")
    os.makedirs(carpeta)
    with open(os.path.join(carpeta, "config.txt"), "w", encoding="utf-8") as f:
        f.write("notebook : " + str(nombre) + "\n")
        f.write("fecha    : " + _dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + "\n")
        if config:
            for k, v in config.items():
                f.write(str(k) + " : " + str(v) + "\n")
    print("[Ejecucion " + str(n) + "] Resultados en: " + carpeta)
    return carpeta

def log(texto, carpeta):
    print(texto)
    with open(os.path.join(carpeta, "resultados.txt"), "a", encoding="utf-8") as f:
        f.write(str(texto) + "\n")

def guardar_csv(resultados, ruta):
    if not resultados: return
    cols = list(resultados[0].keys())
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(",".join(cols) + "\n")
        for r in resultados:
            f.write(",".join(str(r[c]) for c in cols) + "\n")


In [ ]:
import numpy as np, os, time
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from sklearn.metrics import (recall_score,f1_score,roc_auc_score,average_precision_score,
                             accuracy_score,precision_recall_curve,confusion_matrix)
import matplotlib.pyplot as plt
gpus=tf.config.list_physical_devices("GPU")
for g in gpus: tf.config.experimental.set_memory_growth(g,True)
print(f"GPUs: {len(gpus)}" if gpus else "Sin GPU, se usara CPU")
DATOS="datos_procesados/ulb.npz"; MODELO="cnn"; CORRIDAS=5; EPOCAS=50; BATCH=2048; BALANCEO=True

In [ ]:
def construir(nf):
    m=models.Sequential([
        layers.Input(shape=(nf,1)),
        layers.Conv1D(64,3,activation="relu",padding="same"),
        layers.MaxPooling1D(2),
        layers.Conv1D(32,3,activation="relu",padding="same"),
        layers.GlobalMaxPooling1D(),
        layers.Dense(32,activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1,activation="sigmoid")])
    m.compile(optimizer="adam",loss="binary_crossentropy"); return m

In [ ]:
carpeta=iniciar_ejecucion(f"04_{MODELO}",{"datos":DATOS,"modelo":MODELO,"corridas":CORRIDAS,"balanceo":BALANCEO})
d=np.load(DATOS,allow_pickle=True)
Xtr,ytr=d["X_train"],d["y_train"].astype(np.float32)
Xte,yte=d["X_test"],d["y_test"].astype(np.float32)
nf=Xtr.shape[1]
if MODELO=="cnn": Xtr=Xtr.reshape(-1,nf,1); Xte=Xte.reshape(-1,nf,1)
cw=None
if BALANCEO:
    nn,npo=(ytr==0).sum(),(ytr==1).sum(); cw={0:len(ytr)/(2*nn),1:len(ytr)/(2*npo)}
    log(f"class_weight={cw}",carpeta)
metricas=[]
for c in range(CORRIDAS):
    tf.keras.utils.set_random_seed(42+c)
    modelo=construir(nf)
    es=tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)
    t0=time.perf_counter()
    h=modelo.fit(Xtr,ytr,validation_split=0.1,epochs=EPOCAS,batch_size=BATCH,class_weight=cw,callbacks=[es],verbose=2 if c==0 else 0)
    t_ent=time.perf_counter()-t0
    prob=modelo.predict(Xte,batch_size=4096,verbose=0).ravel(); pred=(prob>=0.5).astype(int)
    m={"accuracy":accuracy_score(yte,pred),"recall":recall_score(yte,pred,zero_division=0),
       "f1":f1_score(yte,pred,zero_division=0),"auc_roc":roc_auc_score(yte,prob),
       "auc_pr":average_precision_score(yte,prob),"tiempo_s":t_ent}
    metricas.append(m)
    log(f"Corrida {c+1}/{CORRIDAS}: "+"  ".join(f"{k}={v:.4f}" for k,v in m.items()),carpeta)
    if c==0:
        pr,rc,_=precision_recall_curve(yte,prob)
        plt.figure(figsize=(7,5)); plt.plot(rc,pr,lw=2,label=f"AUC-PR={m['auc_pr']:.3f}")
        plt.xlabel("Recall"); plt.ylabel("Precision"); plt.legend(); plt.grid(alpha=0.3)
        plt.title(f"Curva PR - {MODELO.upper()}")
        plt.savefig(os.path.join(carpeta,"curva_pr.png"),dpi=150,bbox_inches="tight"); plt.show()
        cm=confusion_matrix(yte,pred); log(f"Matriz de confusion:\n{cm}",carpeta)
        plt.figure(figsize=(5,5)); plt.imshow(cm,cmap=plt.cm.Blues)
        for i in range(2):
            for j in range(2):
                plt.text(j,i,f"{cm[i,j]}",ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black")
        plt.xticks([0,1],["No Fraude","Fraude"]); plt.yticks([0,1],["No Fraude","Fraude"])
        plt.xlabel("Prediccion"); plt.ylabel("Real"); plt.title(f"Matriz - {MODELO.upper()}")
        plt.savefig(os.path.join(carpeta,"matriz.png"),dpi=150,bbox_inches="tight"); plt.show()
    plt.figure(figsize=(7,4)); plt.plot(h.history["loss"],label="Entrenamiento"); plt.plot(h.history["val_loss"],label="Validacion")
    plt.xlabel("Epoca"); plt.ylabel("Perdida"); plt.legend(); plt.grid(alpha=0.3); plt.title(f"Perdida - {MODELO.upper()} (corrida {c+1})")
    plt.savefig(os.path.join(carpeta,f"perdida_c{c+1}.png"),dpi=150,bbox_inches="tight"); plt.close()
log(f"\n=== RESUMEN (media +/- desv, {CORRIDAS} corridas) - {MODELO.upper()} ===",carpeta)
for k in metricas[0]:
    v=[mm[k] for mm in metricas]; log(f"{k:10s}: {np.mean(v):.4f} +/- {np.std(v):.4f}",carpeta)
guardar_csv(metricas,os.path.join(carpeta,"metricas_por_corrida.csv"))
log(f"\nGuardado en: {carpeta}",carpeta)